In [3]:
# 04_modeling_experiments.ipynb

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve

# --------------------------------------------------
# Load processed features 
# --------------------------------------------------
BASE_DIR = Path.cwd().parent
TRAIN_CSV = BASE_DIR / "data" / "processed" / "train_features_final.csv"
EVAL_CSV = BASE_DIR / "data" / "processed" / "eval_features_final.csv"

train_df = pd.read_csv(TRAIN_CSV)
eval_df = pd.read_csv(EVAL_CSV)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

# --------------------------------------------------
# Split train into X/y
# --------------------------------------------------
target_col = "TARGET"
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_eval = eval_df.drop(columns=[target_col])
y_eval = eval_df[target_col]

# --------------------------------------------------
# Train/validation split for experiments
# --------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
print(f"Train split: {X_tr.shape}, Validation split: {X_val.shape}")

# --------------------------------------------------
# Define models to try
# --------------------------------------------------
models = {
    "logistic_regression": LogisticRegression(max_iter=500, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "xgboost": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric="logloss", random_state=42)
}

# --------------------------------------------------
# Function to compute KS
# --------------------------------------------------
def ks_metric(y_true, y_pred_proba):
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    return max(tpr - fpr)

# --------------------------------------------------
# Train and evaluate each model
# --------------------------------------------------
model_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_tr, y_tr)

    # Predict probabilities on validation
    if hasattr(model, "predict_proba"):
        y_val_pred = model.predict_proba(X_val)[:, 1]
    else:
        y_val_pred = model.predict(X_val)

    auc = roc_auc_score(y_val, y_val_pred)
    ks = ks_metric(y_val, y_val_pred)

    model_results[name] = {"model": model, "auc": auc, "ks": ks}
    print(f"{name} | AUC: {auc:.4f} | KS: {ks:.4f}")

# --------------------------------------------------
# Select champion
# --------------------------------------------------
champion_name = max(model_results, key=lambda k: model_results[k]["ks"])
champion_model = model_results[champion_name]["model"]
print(f"\nChampion model: {champion_name} | KS: {model_results[champion_name]['ks']:.4f}")


Train shape: (246008, 60)
Eval shape: (61503, 60)
Train split: (196806, 59), Validation split: (49202, 59)

Training logistic_regression...
logistic_regression | AUC: 0.7253 | KS: 0.3372

Training random_forest...
random_forest | AUC: 0.6682 | KS: 0.2481

Training xgboost...


c:\Users\Admin\Desktop\CREDIT_RISK_SCORING\env2\Lib\site-packages\xgboost\training.py:199: UserWarning: [14:55:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


xgboost | AUC: 0.7019 | KS: 0.3011

Champion model: logistic_regression | KS: 0.3372
